In [44]:
USE WideWorldImporters
SELECT StockItemID, StockItemName, UnitPrice
FROM Warehouse.StockItems
WHERE StockItemID = (
    SELECT TOP 1 StockItemID
    FROM Sales.OrderLines
    WHERE OrderID IN (
        SELECT OrderID 
        FROM Sales.Orders 
        WHERE OrderDate >= DATEADD(YEAR, -9, GETDATE())
    )
    ORDER BY UnitPrice DESC
);

(1 row affected)

Total execution time: 00:00:00.109

StockItemID,StockItemName,UnitPrice
215,Air cushion machine (Blue),1899.00


**1\. This query finds the most expensive item sold in the last 9 years.**

<span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">• It uses a scalar subquery because it returns only one value (a single StockItemID).</span>  
<span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">• The subquery first finds orders from the last 9 years, then gets items from those orders, and finally selects the most expensive one.</span>  
<span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">• This is important for pricing strategies and knowing which items bring the highest revenue.</span>

**2\. This query finds customers who have never placed an order.**

<span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">• It uses EXISTS/NOT EXISTS, which is useful for checking if a related record exists in another table.</span>  
<span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">• The subquery checks for matching orders in the Sales.Orders table. If no match is found, the customer is included in the result.</span>  
<span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">• This is important because, this helps businesses identify inactive customers and target them for promotions or follow-ups.</span>

In [47]:
 USE WideWorldImporters
 SELECT CustomerID, CustomerName
FROM Sales.Customers C
WHERE NOT EXISTS (
    SELECT 1 
    FROM Sales.Orders O
    WHERE O.CustomerID = C.CustomerID
);

(0 rows affected)

Total execution time: 00:00:00.017

CustomerID,CustomerName


**3\. This query finds the top 3 salespeople who processed the most orders.**

<span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">•</span> <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">It uses a </span> <span class="s2" style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">subquery</span> <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);"> to count orders for each salesperson and selects the top 3.</span>  
<span class="s1" style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">•<span class="Apple-tab-span"> </span>The main query </span> <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">joins the salesperson’s ID with their full name</span> <span class="s1" style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);"> from the </span> <span class="s3" style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">Application.People</span> <span class="s1" style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);"> table.</span>  
<span class="s1" style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">•<span class="Apple-tab-span"> </span>This helps the company </span> <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">recognize top performers and reward them</span><span class="s1" style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">, or </span> <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">analyze sales strategies for improvement</span><span class="s1" style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">.</span>

In [48]:
USE WideWorldImporters;
SELECT p.PersonID, p.FullName, OrderCount
FROM Application.People p
JOIN (
    SELECT TOP 3 SalespersonPersonID, COUNT(OrderID) AS OrderCount
    FROM Sales.Orders
    GROUP BY SalespersonPersonID
    ORDER BY OrderCount DESC
) AS OrderCounts
ON p.PersonID = OrderCounts.SalespersonPersonID
ORDER BY OrderCount DESC;

(3 rows affected)

Total execution time: 00:00:00.031

PersonID,FullName,OrderCount
16,Archer Lamble,7532
2,Kayla Woodcock,7474
13,Hudson Hollinworth,7400


**<span class="s1" style="color: var(--vscode-foreground); font-family: -apple-system, BlinkMacSystemFont, sans-serif;">4. This query </span> <span style="color: var(--vscode-foreground); font-family: -apple-system, BlinkMacSystemFont, sans-serif;">retrieves the most recent order for each customer</span><span class="s1" style="color: var(--vscode-foreground); font-family: -apple-system, BlinkMacSystemFont, sans-serif;">.</span>**  
<span class="s1" style="color: var(--vscode-foreground); font-family: -apple-system, BlinkMacSystemFont, sans-serif;">•<span class="Apple-tab-span"> </span>It uses a </span> <span style="color: var(--vscode-foreground); font-family: -apple-system, BlinkMacSystemFont, sans-serif;">correlated subquery</span><span class="s1" style="color: var(--vscode-foreground); font-family: -apple-system, BlinkMacSystemFont, sans-serif;">, meaning it </span> <span style="color: var(--vscode-foreground); font-family: -apple-system, BlinkMacSystemFont, sans-serif;">checks each row against a related set of data</span><span class="s1" style="color: var(--vscode-foreground); font-family: -apple-system, BlinkMacSystemFont, sans-serif;">.</span>  
<span style="color: var(--vscode-foreground); font-family: -apple-system, BlinkMacSystemFont, sans-serif;">•</span> <span style="color: var(--vscode-foreground); font-family: -apple-system, BlinkMacSystemFont, sans-serif;">The subquery finds the </span> <span class="s2" style="color: var(--vscode-foreground); font-family: -apple-system, BlinkMacSystemFont, sans-serif;">latest order date</span> <span style="color: var(--vscode-foreground); font-family: -apple-system, BlinkMacSystemFont, sans-serif;"> for each </span> <span class="s3" style="color: var(--vscode-foreground); font-family: -apple-system, BlinkMacSystemFont, sans-serif;">CustomerID</span><span style="color: var(--vscode-foreground); font-family: -apple-system, BlinkMacSystemFont, sans-serif;">, and the main query </span> <span class="s2" style="color: var(--vscode-foreground); font-family: -apple-system, BlinkMacSystemFont, sans-serif;">retrieves the matching order details</span><span style="color: var(--vscode-foreground); font-family: -apple-system, BlinkMacSystemFont, sans-serif;">.</span>  
<span style="color: var(--vscode-foreground); font-family: -apple-system, BlinkMacSystemFont, sans-serif;">•</span> <span style="color: var(--vscode-foreground); font-family: -apple-system, BlinkMacSystemFont, sans-serif;">This is </span> <span class="s2" style="color: var(--vscode-foreground); font-family: -apple-system, BlinkMacSystemFont, sans-serif;">important for businesses to track recent customer activity</span><span style="color: var(--vscode-foreground); font-family: -apple-system, BlinkMacSystemFont, sans-serif;">, analyze purchasing patterns, and improve customer engagement.</span>

In [29]:
USE WideWorldImporters
SELECT CustomerID, OrderID, OrderDate
FROM Sales.Orders AS O1
WHERE OrderDate = (
    SELECT MAX(O2.OrderDate) 
    FROM Sales.Orders AS O2 
    WHERE O2.CustomerID = O1.CustomerID
);

(785 rows affected)

Total execution time: 00:00:00.116

CustomerID,OrderID,OrderDate
905,69549,2016-03-31
905,69588,2016-03-31
66,70159,2016-04-11
128,70223,2016-04-12
128,70240,2016-04-12
15,70298,2016-04-13
15,70300,2016-04-13
131,70324,2016-04-13
130,70330,2016-04-13
131,70352,2016-04-13


5\. **This query** **identifies the three salespeople who processed the least number of orders.**

<span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">•</span> <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">It uses a </span> <span class="s2" style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">subquery</span> <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);"> to find the </span> <span class="s2" style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">SalespersonPersonID with the lowest order count</span> <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);"> and matches them with their names from the </span> <span class="s3" style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">Application.People</span> <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);"> table.</span>  
<span class="s1" style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">•<span class="Apple-tab-span"> </span>The </span> <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">main query retrieves and displays the salesperson’s ID, name, and total orders</span><span class="s1" style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">.</span>  
<span class="s1" style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">•<span class="Apple-tab-span"> </span>This information is </span> <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">important for management to evaluate performance and provide training or support where needed</span><span class="s1" style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">.</span>

In [49]:
USE WideWorldImporters
SELECT PersonID, FullName, OrderCount
FROM (
    SELECT SalespersonPersonID AS PersonID, 
           p.FullName,
           COUNT(OrderID) AS OrderCount
    FROM Sales.Orders o
    JOIN Application.People p ON o.SalespersonPersonID = p.PersonID
    GROUP BY SalespersonPersonID, p.FullName
) AS OrderCounts
WHERE PersonID IN (
    SELECT TOP 3 SalespersonPersonID
    FROM Sales.Orders
    GROUP BY SalespersonPersonID
    ORDER BY COUNT(OrderID) ASC
)
ORDER BY OrderCount ASC;

(3 rows affected)

Total execution time: 00:00:00.082

PersonID,FullName,OrderCount
8,Anthony Grosse,7257
14,Lily Code,7268
7,Amy Trefl,7276


\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_Chapter 5 \_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_

6. **This query** **finds all invoices issued on the last day of each month.**

<span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">•</span> <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">It uses a </span> <span class="s2" style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">Common Table Expression (CTE)</span> <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);"> to create a temporary table (</span><span class="s3" style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">LastDayInvoices</span><span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">) that calculates the </span> <span class="s2" style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">last day of each invoice’s month</span> <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);"> using </span> <span class="s3" style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">EOMONTH()</span><span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">.</span>  
<span class="s1" style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">•<span class="Apple-tab-span"> </span>The </span> <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">main query filters invoices where the invoice date matches the last day of the month</span><span class="s1" style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">.</span>  
<span class="s1" style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">•This</span> <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">&nbsp;is this important because, it&nbsp;</span> <span class="s1" style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">Helps </span> <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">track end-of-month sales trends</span><span class="s1" style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">. and also&nbsp;</span> <span class="s1" style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">Useful for </span> <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">financial reporting and closing balances</span><span class="s1" style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">.</span>

In [38]:
USE WideWorldImporters;

WITH LastDayInvoices AS (
    SELECT InvoiceID, CustomerID, InvoiceDate,
           EOMONTH(InvoiceDate) AS LastDayOfMonth
    FROM Sales.Invoices
)
SELECT InvoiceID, CustomerID, InvoiceDate
FROM LastDayInvoices
WHERE InvoiceDate = LastDayOfMonth;

(2238 rows affected)

Total execution time: 00:00:00.269

InvoiceID,CustomerID,InvoiceDate
1574,479,2013-01-31
1575,184,2013-01-31
1576,941,2013-01-31
1577,426,2013-01-31
1578,807,2013-01-31
1579,571,2013-01-31
1580,190,2013-01-31
1581,98,2013-01-31
1582,599,2013-01-31
1583,478,2013-01-31


**7\. It counts the number of orders placed each month in each year.**

<span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">• </span> <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">The inner query extracts the year and month from OrderDate and creates a temporary table (MonthlyOrders).</span>  
<span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">• </span> <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">The outer query groups the data by year and month to count total orders.</span>  
 <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">&nbsp;this is important because it&nbsp;</span> <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">Helps identify seasonal trends in sales.</span>  
<span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">• </span> <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">Useful for forecasting demand and making business decisions.</span>

In [39]:
USE WideWorldImporters;

SELECT OrderYear, OrderMonth, COUNT(*) AS TotalOrders
FROM (
    SELECT YEAR(OrderDate) AS OrderYear, 
           MONTH(OrderDate) AS OrderMonth, 
           OrderID
    FROM Sales.Orders
) AS MonthlyOrders
GROUP BY OrderYear, OrderMonth
ORDER BY OrderYear, OrderMonth;

(41 rows affected)

Total execution time: 00:00:00.108

OrderYear,OrderMonth,TotalOrders
2013,1,1674
2013,2,1139
2013,3,1683
2013,4,1696
2013,5,1808
2013,6,1675
2013,7,1886
2013,8,1537
2013,9,1617
2013,10,1618


8. **It retrieves all orders processed by a specific salesperson (Employee ID 3).**

<span class="s1" style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">•<span class="Apple-tab-span"> </span></span> <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">A Common Table Expression (CTE) named EmployeeOrders is used to filter orders by the salesperson’s ID.</span>  
<span class="s1" style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">•<span class="Apple-tab-span"> </span></span> <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">The main query selects and orders these records by OrderDate in descending order.</span>  
<span class="s1" style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">•<span class="Apple-tab-span"> </span></span> <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">Helps track an employee’s sales performance over time.</span>  
<span class="s1" style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">•<span class="Apple-tab-span"> </span></span> <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">Useful for performance evaluation and setting sales targets</span>

In [40]:
USE WideWorldImporters;
DECLARE @EmpID INT = 3;

WITH EmployeeOrders AS (
    SELECT OrderID, CustomerID, OrderDate
    FROM Sales.Orders
    WHERE SalespersonPersonID = @EmpID
)
SELECT OrderID, CustomerID, OrderDate
FROM EmployeeOrders
ORDER BY OrderDate DESC;

(7281 rows affected)

Displaying Top 5000 rows.

Total execution time: 00:00:00.054

OrderID,CustomerID,OrderDate
73517,1018,2016-05-31
73529,466,2016-05-31
73540,835,2016-05-31
73542,119,2016-05-31
73546,810,2016-05-31
73548,840,2016-05-31
73574,838,2016-05-31
73576,915,2016-05-31
73582,498,2016-05-31
73587,1018,2016-05-31


**9\. Counts unique customers placing orders each year and tracks growth.**

<span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">• </span> <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">Uses a derived table (YearlyCustomers) to count customers per year.</span>  
<span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">• </span> <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">The LAG() function compares the current year’s count with the previous year.</span>  
<span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">• </span> <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">Helps understand customer retention and business growth.</span>  
<span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">• </span> <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">Useful for planning strategies to attract and keep customers.</span>

In [41]:
USE WideWorldImporters;
SELECT OrderYear, CustomerCount,
       LAG(CustomerCount, 1, 0) OVER (ORDER BY OrderYear) AS PreviousYearCustomers,
       (CustomerCount - LAG(CustomerCount, 1, 0) OVER (ORDER BY OrderYear)) AS Growth
FROM (
    SELECT YEAR(OrderDate) AS OrderYear, COUNT(DISTINCT CustomerID) AS CustomerCount
    FROM Sales.Orders
    GROUP BY YEAR(OrderDate)
) AS YearlyCustomers;

(4 rows affected)

Total execution time: 00:00:00.070

OrderYear,CustomerCount,PreviousYearCustomers,Growth
2013,625,0,625
2014,640,625,15
2015,657,640,17
2016,663,657,6


_10\. Identify salespersons who processed orders in consecutive years._

<span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">• It </span> <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">groups orders by salesperson and year</span> <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);"> using a Common Table Expression (CTE).</span>  
<span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">• Then, it </span> <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">joins the data to itself</span> <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);"> to check if the same salesperson had orders in the following year.</span>  
_Why :_<span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">This query helps </span> <span class="s1" style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">identify consistent salespersons</span> <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">who contribute to the company’s long-term success. It can be used for </span> <span class="s1" style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">employee performance analysis and retention strategies</span><span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">.</span>

In [50]:
USE WideWorldImporters;
WITH YearlyEmployeeOrders AS (
    SELECT SalespersonPersonID, YEAR(OrderDate) AS OrderYear
    FROM Sales.Orders
    GROUP BY SalespersonPersonID, YEAR(OrderDate)
)
SELECT Cur.SalespersonPersonID, Cur.OrderYear, Prv.OrderYear AS PreviousYear
FROM YearlyEmployeeOrders AS Cur
JOIN YearlyEmployeeOrders AS Prv
ON Cur.SalespersonPersonID = Prv.SalespersonPersonID
AND Cur.OrderYear = Prv.OrderYear + 1;

(30 rows affected)

Total execution time: 00:00:00.101

SalespersonPersonID,OrderYear,PreviousYear
14,2016,2015
2,2016,2015
16,2016,2015
6,2015,2014
20,2015,2014
8,2015,2014
13,2015,2014
14,2014,2013
15,2015,2014
2,2014,2013
